In [1]:
import pandas as pd
from datetime import datetime
from etl.esios_provider import EsiosAPI

In [2]:
start_date = datetime(2018, 10,1,0,0)
end_date = datetime(2026,1,30,23,55)
key = 'abc40f97d97902794c193210375c8b8bd9158f9bc7dc619b6f6fa7234917dea4'

In [3]:
api = EsiosAPI(key=key)

In [4]:
df_spot_h = api._get_chunks(indicator=600, start_date=start_date, end_date=end_date)

Retrieving data for 600 from 2018-10-01 to 2018-10-31
Retrieving data for 600 from 2018-10-31 to 2018-11-30
Retrieving data for 600 from 2018-11-30 to 2018-12-30
Retrieving data for 600 from 2018-12-30 to 2019-01-29
Retrieving data for 600 from 2019-01-29 to 2019-02-28
Retrieving data for 600 from 2019-02-28 to 2019-03-30
Retrieving data for 600 from 2019-03-30 to 2019-04-29
Retrieving data for 600 from 2019-04-29 to 2019-05-29
Retrieving data for 600 from 2019-05-29 to 2019-06-28
Retrieving data for 600 from 2019-06-28 to 2019-07-28
Retrieving data for 600 from 2019-07-28 to 2019-08-27
Retrieving data for 600 from 2019-08-27 to 2019-09-26
Retrieving data for 600 from 2019-09-26 to 2019-10-26
Retrieving data for 600 from 2019-10-26 to 2019-11-25
Retrieving data for 600 from 2019-11-25 to 2019-12-25
Retrieving data for 600 from 2019-12-25 to 2020-01-24
Retrieving data for 600 from 2020-01-24 to 2020-02-23
Retrieving data for 600 from 2020-02-23 to 2020-03-24
Retrieving data for 600 from

In [5]:
df_spot_h.to_pickle(f'spot_raw_{start_date.strftime("%Y%m%d")}_{end_date.strftime("%Y%m%d")}.pkl')

In [14]:
ds = df_spot_h[df_spot_h['datetime'].dt.year == 2025].groupby([df_spot_h['geo_name'], df_spot_h['datetime'].dt.year, df_spot_h['datetime'].dt.month])['value'].mean()
ds.index.set_names(['zone', 'year', 'month'], inplace=True)

df_global = ds.reset_index()
# df_global_melted = df_global.melt(id_vars=['zone', 'year'], value_vars=['negative', 'zero_negative', 'zero'], var_name='value_type', value_name='value')
df_global_pivot = df_global.pivot_table(index=['year', 'month'], columns='zone', values='value').reset_index()
df_global_pivot.to_excel('spot_monthly.xlsx', index=False)
df_global_pivot

zone,year,month,Alemania,Bélgica,España,Francia,Países Bajos,Portugal
0,2025,1,114.140161,111.153139,96.689839,102.274704,116.699778,96.731452
1,2025,2,128.522366,128.749048,108.307827,122.658482,125.825074,108.215982
2,2025,3,94.727497,91.205491,53.094509,76.878560,91.721144,52.597658
3,2025,4,77.935653,73.385597,26.809139,42.210736,74.844306,25.906431
4,2025,5,67.338629,61.069449,16.927527,19.376196,64.207124,25.793817
5,2025,6,63.987500,65.325500,72.597208,40.742222,68.376437,74.173167
6,2025,7,87.795228,83.070712,70.014651,57.977513,87.530551,70.103387
7,2025,8,76.990255,68.979973,68.444691,54.439758,74.862417,68.675847
8,2025,9,83.511083,63.604708,61.042333,34.808056,77.649653,61.189736
9,2025,10,84.996634,75.076091,75.749443,57.472322,82.345309,76.467527


In [19]:
ds_neagative = df_spot_h[df_spot_h['value'] < 0].groupby([df_spot_h['geo_name'], df_spot_h['datetime'].dt.year])['value'].count()
ds_neagative.index.set_names(['zone', 'year'], inplace=True)
ds_zero_neg = df_spot_h[df_spot_h['value'] <= 0].groupby([df_spot_h['geo_name'], df_spot_h['datetime'].dt.year])['value'].count()
ds_zero_neg.index.set_names(['zone', 'year'], inplace=True)
ds_zero = df_spot_h[df_spot_h['value'] == 0].groupby([df_spot_h['geo_name'], df_spot_h['datetime'].dt.year])['value'].count()
ds_zero.index.set_names(['zone', 'year'], inplace=True)

ds_global = pd.concat([ds_neagative, ds_zero_neg, ds_zero], axis=1)
ds_global.columns = ['negative', 'zero_negative', 'zero']
df_global = ds_global.reset_index()
df_global_melted = df_global.melt(id_vars=['zone', 'year'], value_vars=['negative', 'zero_negative', 'zero'], var_name='value_type', value_name='value')
df_global_pivot = df_global_melted.pivot_table(index=['zone', 'value_type'], columns='year', values='value').reset_index()
df_global_pivot.to_excel('negative_periods.xlsx', index=False)
df_global_pivot

year,zone,value_type,2019,2020,2021,2022,2023,2024,2025,2026
0,Alemania,negative,NaN,170.0,139.0,69.0,301.0,448.0,571.0,2.0
1,Alemania,zero,NaN,1.0,7.0,6.0,24.0,59.0,78.0,NaN
2,Alemania,zero_negative,NaN,171.0,146.0,75.0,325.0,507.0,649.0,2.0
3,Bélgica,negative,NaN,118.0,159.0,111.0,216.0,404.0,519.0,NaN
4,Bélgica,zero,NaN,4.0,8.0,29.0,53.0,53.0,56.0,NaN
5,Bélgica,zero_negative,NaN,122.0,167.0,140.0,269.0,457.0,575.0,NaN
6,España,negative,NaN,NaN,NaN,NaN,NaN,247.0,556.0,NaN
7,España,zero,NaN,NaN,NaN,3.0,109.0,537.0,241.0,3.0
8,España,zero_negative,NaN,NaN,NaN,3.0,109.0,784.0,797.0,3.0
9,Francia,negative,27.0,102.0,64.0,4.0,147.0,352.0,513.0,NaN


In [18]:
df_spot_h[(df_spot_h['geo_name'] == 'España') & (df_spot_h['datetime'].dt.year == 2025) & (df_spot_h['value'] < 0)]['value'].count()

np.int64(556)

In [5]:
# df_spot = api.get_data(type="spot", start_date=start_date, end_date=end_date)
# df_gen = api.get_data(type="real_generation", start_date=start_date, end_date=end_date)
df_demanda = api.get_data(type="real_demand", start_date=start_date, end_date=end_date)
# df_links = api.get_data(type="links", start_date=start_date, end_date=end_date)

Retrieving data for 600 from 2024-10-01 to 2024-10-31
Retrieving data for 600 from 2024-10-31 to 2024-11-30
Retrieving data for 600 from 2024-11-30 to 2024-12-30
Retrieving data for 600 from 2024-12-30 to 2025-01-29
Retrieving data for 600 from 2025-01-29 to 2025-02-28
Retrieving data for 600 from 2025-02-28 to 2025-03-30
Retrieving data for 600 from 2025-03-30 to 2025-04-29
Retrieving data for 600 from 2025-04-29 to 2025-05-29
Retrieving data for 600 from 2025-05-29 to 2025-06-28
Retrieving data for 600 from 2025-06-28 to 2025-07-28
Retrieving data for 600 from 2025-07-28 to 2025-08-27
Retrieving data for 600 from 2025-08-27 to 2025-09-26
Retrieving data for 600 from 2025-09-26 to 2025-09-30
Retrieving data for 1293 from 2024-10-01 to 2024-10-31
Retrieving data for 1293 from 2024-10-31 to 2024-11-30
Retrieving data for 1293 from 2024-11-30 to 2024-12-30
Retrieving data for 1293 from 2024-12-30 to 2025-01-29
Retrieving data for 1293 from 2025-01-29 to 2025-02-28
Retrieving data for 129

In [ ]:
df_demanda['load_shifted'] = df_demanda['load'].shift(1)
df_demanda['hour_shifted'] = df_demanda['datetime'].shift(1)
df_demanda['hour_gap'] = df_demanda['hour_shifted'].dt.hour.astype(str) + '-' + df_demanda['hour'].astype(str)
df_demanda['gap'] = df_demanda['load'] - df_demanda['load_shifted']
df_demanda['abs_gap'] = df_demanda['gap'].abs()
df_demanda.dropna(subset=['gap'], inplace=True)
df_demanda.groupby([df_demanda['datetime'].dt.year])['abs_gap'].describe()

,datetime,Importación Francia,Importación Portugal,Importación Marruecos,Importación Andorra,Exportación Francia,Exportación Portugal,Exportación Marruecos,Exportación Andorra
0,2019-01-01 00:00:00,2468.887,428.808,180.900,0.0,-11.005,-413.212,-0.200,-30.58
1,2019-01-01 01:00:00,2419.963,208.358,378.000,0.0,-1.345,-948.936,0.000,-28.35
2,2019-01-01 02:00:00,2421.463,87.870,410.600,0.0,-0.133,-1037.396,0.000,-26.05
3,2019-01-01 03:00:00,2406.028,8.957,399.500,0.0,-0.102,-1392.104,0.000,-24.41
4,2019-01-01 04:00:00,2447.643,0.000,423.500,0.0,-0.102,-1988.196,0.000,-23.36
...,...,...,...,...,...,...,...,...,...
52603,2024-12-31 19:00:00,3400.766,273.672,0.000,0.0,-193.723,-1700.463,-191.376,-53.96
52604,2024-12-31 20:00:00,3413.594,256.824,0.000,0.0,-143.105,-1955.914,-181.008,-51.43
52605,2024-12-31 21:00:00,3165.059,180.144,27.864,0.0,-16.344,-1743.021,-44.496,-45.22
52606,2024-12-31 22:00:00,2910.511,8.640,85.752,0.0,-35.545,-2197.923,-0.216,-64.88
